In [3]:
import pandas as pd
import re
from pathlib import Path

def extract_sample_titles_from_soft(file_path):
    """
    Extract sample titles from a GEO SOFT file.
    
    Parameters:
    file_path (str): Path to the SOFT file
    
    Returns:
    list: List of sample titles
    """
    sample_titles = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                # Look for lines that contain sample title information
                if line.startswith('!Sample_title'):
                    # Extract the title part after the = sign
                    title = line.split('=', 1)[1].strip() if '=' in line else line.replace('!Sample_title', '').strip()
                    sample_titles.append(title)
                    
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []
    
    return sample_titles

def display_sample_info(sample_titles):
    """
    Display sample titles in a formatted way and create a DataFrame.
    
    Parameters:
    sample_titles (list): List of sample titles
    
    Returns:
    pandas.DataFrame: DataFrame with sample information
    """
    if not sample_titles:
        print("No sample titles found.")
        return pd.DataFrame()
    
    print(f"Found {len(sample_titles)} samples:")
    print("=" * 50)
    
    # Create a DataFrame for better display
    df = pd.DataFrame({
        'Sample_ID': [f'Sample_{i+1}' for i in range(len(sample_titles))],
        'Sample_Title': sample_titles
    })
    
    # Display numbered list
    for i, title in enumerate(sample_titles, 1):
        print(f"{i:2d}. {title}")
    
    print("=" * 50)
    print(f"\nSample titles extracted successfully!")
    
    return df

# Main execution
def main():
    # Specify the path to your SOFT file
    soft_file_path = "GSE45827_family.soft"  # Update this path as needed
    
    print(f"Processing SOFT file: {soft_file_path}")
    print("-" * 50)
    
    # Extract sample titles
    sample_titles = extract_sample_titles_from_soft(soft_file_path)
    
    # Display results
    df_samples = display_sample_info(sample_titles)
    
    # Save to CSV if samples found
    if not df_samples.empty:
        output_file = "sample_titles.csv"
        df_samples.to_csv(output_file, index=False)
        print(f"\nSample titles saved to: {output_file}")
        
        # Display DataFrame
        print("\nDataFrame preview:")
        print(df_samples.head(10))  # Show first 10 rows
        
        if len(df_samples) > 10:
            print(f"... and {len(df_samples) - 10} more samples")
    
    return df_samples

# Alternative function for more detailed extraction
def extract_detailed_sample_info(file_path):
    """
    Extract more detailed sample information including other metadata.
    
    Parameters:
    file_path (str): Path to the SOFT file
    
    Returns:
    pandas.DataFrame: DataFrame with detailed sample information
    """
    samples_data = []
    current_sample = {}
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                
                # Start of a new sample
                if line.startswith('^SAMPLE'):
                    # Save previous sample if it exists
                    if current_sample:
                        samples_data.append(current_sample.copy())
                        current_sample = {}
                    # Extract sample ID
                    sample_id = line.split('=')[1].strip() if '=' in line else ''
                    current_sample['Sample_ID'] = sample_id
                
                # Extract sample metadata
                elif line.startswith('!Sample_title'):
                    current_sample['Title'] = line.split('=', 1)[1].strip() if '=' in line else ''
                elif line.startswith('!Sample_source_name'):
                    current_sample['Source'] = line.split('=', 1)[1].strip() if '=' in line else ''
                elif line.startswith('!Sample_organism'):
                    current_sample['Organism'] = line.split('=', 1)[1].strip() if '=' in line else ''
                elif line.startswith('!Sample_characteristics'):
                    # Handle multiple characteristics
                    char = line.split('=', 1)[1].strip() if '=' in line else ''
                    if 'Characteristics' not in current_sample:
                        current_sample['Characteristics'] = []
                    current_sample['Characteristics'].append(char)
            
            # Don't forget the last sample
            if current_sample:
                samples_data.append(current_sample)
                
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading file: {e}")
        return pd.DataFrame()
    
    # Convert to DataFrame
    if samples_data:
        # Handle lists in characteristics
        for sample in samples_data:
            if 'Characteristics' in sample and isinstance(sample['Characteristics'], list):
                sample['Characteristics'] = '; '.join(sample['Characteristics'])
        
        df = pd.DataFrame(samples_data)
        return df
    else:
        return pd.DataFrame()

# Execute the main function
if __name__ == "__main__":
    # Basic extraction
    df_basic = main()
    
    print("\n" + "="*70)
    print("DETAILED EXTRACTION (including additional metadata)")
    print("="*70)
    
    # Detailed extraction
    df_detailed = extract_detailed_sample_info("GSE45827_family.soft")
    
    if not df_detailed.empty:
        print(f"Extracted detailed information for {len(df_detailed)} samples")
        print("\nDetailed sample information:")
        print(df_detailed.head())
        
        # Save detailed info
        df_detailed.to_csv("detailed_sample_info.csv", index=False)
        print(f"\nDetailed sample information saved to: detailed_sample_info.csv")
    else:
        print("No detailed sample information found.")

Processing SOFT file: GSE45827_family.soft
--------------------------------------------------
Found 155 samples:
 1. Basal Sample1 rep1
 2. Basal Sample2 rep1-2
 3. Her2 Sample3 rep1
 4. Basal Sample4 rep1-2
 5. Her2 Sample5 rep1-2
 6. Her2 Sample6 rep1-2
 7. Basal Sample7 rep1
 8. Basal Sample8 rep1-2
 9. Basal Sample9 rep1
10. Basal Sample10 rep1-2
11. Basal Sample11 rep1
12. Her2 Sample12 rep1-2
13. Her2 Sample13 rep1
14. Her2 Sample14 rep1
15. Her2 Sample15 rep1
16. Basal Sample16 rep1-2
17. Her2 Sample17 rep1
18. Basal Sample18 rep1
19. Basal Sample19 rep1-2
20. Her2 Sample20 rep1
21. Her2 Sample21 rep1-2
22. Her2 Sample22 rep1
23. Basal Sample23 rep1-2
24. Her2 Sample24 rep1-2
25. Basal Sample25 rep1
26. Basal Sample26 rep1
27. Basal Sample27 rep1
28. Basal Sample28 rep1
29. Basal Sample29 rep1-2
30. Basal Sample30 rep1
31. Basal Sample31 rep1
32. Basal Sample32 rep1
33. Her2 Sample33 rep1
34. Her2 Sample34 rep1-2
35. Basal Sample35 rep1-2
36. Basal Sample36 rep1-2
37. Basal Samp

In [5]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import defaultdict

def parse_soft_file_for_expression(file_path, sample_prefix):
    """
    Parse SOFT file to extract gene expression data for samples with specific prefix.
    
    Parameters:
    file_path (str): Path to the SOFT file
    sample_prefix (str): Prefix to filter samples (e.g., "Basal ")
    
    Returns:
    tuple: (expression_df, sample_info_df, gene_info_df)
    """
    
    print(f"Parsing SOFT file: {file_path}")
    print(f"Looking for samples with prefix: '{sample_prefix}'")
    print("-" * 60)
    
    # Storage for data
    sample_titles = {}  # sample_id -> title
    sample_info = {}    # sample_id -> metadata dict
    platform_info = {} # platform_id -> probe info
    expression_data = defaultdict(dict)  # sample_id -> {probe_id: expression_value}
    current_platform = None
    current_sample = None
    matching_samples = set()
    
    # Flags for parsing sections
    in_platform_table = False
    in_sample_data = False
    platform_columns = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line_num, line in enumerate(file, 1):
                if line_num % 10000 == 0:
                    print(f"Processing line {line_num}...")
                
                line = line.strip()
                if not line:
                    continue
                
                # Platform section
                if line.startswith('^PLATFORM'):
                    current_platform = line.split('=')[1].strip() if '=' in line else ''
                    in_platform_table = False
                    platform_columns = []
                    print(f"Found platform: {current_platform}")
                
                elif line.startswith('!platform_table_begin'):
                    in_platform_table = True
                    platform_info[current_platform] = {}
                
                elif line.startswith('!platform_table_end'):
                    in_platform_table = False
                
                elif in_platform_table and line.startswith('#'):
                    # Platform table header
                    platform_columns = [col.strip() for col in line[1:].split('\t')]
                
                elif in_platform_table and platform_columns and not line.startswith('!'):
                    # Platform table data
                    values = line.split('\t')
                    if len(values) >= len(platform_columns):
                        probe_id = values[0]
                        platform_info[current_platform][probe_id] = {
                            col: values[i] if i < len(values) else '' 
                            for i, col in enumerate(platform_columns)
                        }
                
                # Sample section
                elif line.startswith('^SAMPLE'):
                    current_sample = line.split('=')[1].strip() if '=' in line else ''
                    sample_info[current_sample] = {'Sample_ID': current_sample}
                    in_sample_data = False
                
                elif current_sample and line.startswith('!Sample_title'):
                    title = line.split('=', 1)[1].strip() if '=' in line else ''
                    sample_titles[current_sample] = title
                    sample_info[current_sample]['Title'] = title
                    
                    # Check if this sample matches our prefix
                    if title.startswith(sample_prefix):
                        matching_samples.add(current_sample)
                        print(f"Found matching sample: {current_sample} - {title}")
                
                elif current_sample and line.startswith('!Sample_'):
                    # Extract other sample metadata
                    key_value = line.split('=', 1)
                    if len(key_value) == 2:
                        key = key_value[0].replace('!Sample_', '').strip()
                        value = key_value[1].strip()
                        sample_info[current_sample][key] = value
                
                elif line.startswith('!sample_table_begin'):
                    in_sample_data = True
                    sample_columns = []
                
                elif line.startswith('!sample_table_end'):
                    in_sample_data = False
                
                elif in_sample_data and current_sample and line.startswith('#'):
                    # Sample data table header
                    sample_columns = [col.strip() for col in line[1:].split('\t')]
                
                elif (in_sample_data and current_sample and sample_columns and 
                      not line.startswith('!') and current_sample in matching_samples):
                    # Sample expression data - only for matching samples
                    values = line.split('\t')
                    if len(values) >= 2:  # At least probe_id and one value
                        probe_id = values[0]
                        # Look for value column (usually "VALUE" or similar)
                        value_col_idx = None
                        for i, col in enumerate(sample_columns):
                            if col.upper() in ['VALUE', 'SIGNAL', 'EXPRESSION']:
                                value_col_idx = i
                                break
                        
                        if value_col_idx is None and len(sample_columns) > 1:
                            value_col_idx = 1  # Default to second column
                        
                        if value_col_idx and value_col_idx < len(values):
                            try:
                                expr_value = float(values[value_col_idx])
                                expression_data[current_sample][probe_id] = expr_value
                            except ValueError:
                                pass  # Skip non-numeric values
    
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return None, None, None
    except Exception as e:
        print(f"Error reading file: {e}")
        return None, None, None
    
    print(f"\nFound {len(matching_samples)} samples with prefix '{sample_prefix}'")
    
    if not matching_samples:
        print("No matching samples found!")
        return None, None, None
    
    # Create expression matrix
    all_probes = set()
    for sample_id in matching_samples:
        all_probes.update(expression_data[sample_id].keys())
    
    all_probes = sorted(list(all_probes))
    print(f"Found {len(all_probes)} probes/genes")
    
    # Build expression dataframe
    expr_matrix = []
    for probe_id in all_probes:
        row = []
        for sample_id in sorted(matching_samples):
            value = expression_data[sample_id].get(probe_id, np.nan)
            row.append(value)
        expr_matrix.append(row)
    
    # Create DataFrames
    sample_cols = [f"{sample_id}_{sample_titles[sample_id]}" for sample_id in sorted(matching_samples)]
    expression_df = pd.DataFrame(expr_matrix, 
                                index=all_probes, 
                                columns=sample_cols)
    
    # Sample info DataFrame
    matching_sample_info = {k: v for k, v in sample_info.items() if k in matching_samples}
    sample_info_df = pd.DataFrame.from_dict(matching_sample_info, orient='index')
    
    # Gene/Probe info DataFrame
    gene_info_data = []
    for platform_id, probes in platform_info.items():
        for probe_id, info in probes.items():
            if probe_id in all_probes:
                info_copy = info.copy()
                info_copy['Platform'] = platform_id
                info_copy['Probe_ID'] = probe_id
                gene_info_data.append(info_copy)
    
    gene_info_df = pd.DataFrame(gene_info_data)
    if not gene_info_df.empty:
        gene_info_df.set_index('Probe_ID', inplace=True)
    
    return expression_df, sample_info_df, gene_info_df

def extract_expression_by_prefix(file_path, sample_prefix, output_prefix="filtered_expression"):
    """
    Main function to extract and save gene expression data for samples with specific prefix.
    
    Parameters:
    file_path (str): Path to the SOFT file
    sample_prefix (str): Prefix to filter samples
    output_prefix (str): Prefix for output files
    
    Returns:
    pandas.DataFrame: Gene expression matrix
    """
    
    # Extract data
    expression_df, sample_info_df, gene_info_df = parse_soft_file_for_expression(file_path, sample_prefix)
    
    if expression_df is None:
        return None
    
    print("\n" + "="*60)
    print("EXTRACTION RESULTS")
    print("="*60)
    
    print(f"Expression matrix shape: {expression_df.shape}")
    print(f"Genes/Probes: {expression_df.shape[0]}")
    print(f"Samples: {expression_df.shape[1]}")
    
    # Display basic statistics
    print(f"\nExpression value statistics:")
    print(expression_df.describe())
    
    # Check for missing values
    missing_count = expression_df.isnull().sum().sum()
    total_values = expression_df.size
    print(f"\nMissing values: {missing_count}/{total_values} ({100*missing_count/total_values:.2f}%)")
    
    # Save files
    expression_file = f"{output_prefix}_matrix.csv"
    sample_info_file = f"{output_prefix}_sample_info.csv"
    gene_info_file = f"{output_prefix}_gene_info.csv"
    
    print(f"\nSaving files...")
    # Make sure to include the index (gene/probe IDs) in the CSV
    expression_df.to_csv(expression_file, index=True)
    print(f"- Expression matrix: {expression_file}")
    
    if not sample_info_df.empty:
        sample_info_df.to_csv(sample_info_file, index=True)
        print(f"- Sample information: {sample_info_file}")
    
    if not gene_info_df.empty:
        gene_info_df.to_csv(gene_info_file, index=True)
        print(f"- Gene/Probe information: {gene_info_file}")
    
    # Display preview
    print(f"\nExpression matrix preview (with gene/probe IDs):")
    print("First few rows and columns:")
    print(expression_df.head(10))
    
    print(f"\nMatrix structure:")
    print(f"- Index (rows): Gene/Probe IDs")
    print(f"- Columns: Sample IDs and titles")
    print(f"- Values: Expression levels")
    
    if expression_df.shape[1] > 5:
        print(f"\nShowing first 5 columns of {expression_df.shape[1]} total columns...")
        print(expression_df.iloc[:10, :5])
    
    return expression_df

# Alternative function for faster processing of large files
def extract_expression_memory_efficient(file_path, sample_prefix, chunk_size=1000):
    """
    Memory-efficient version for very large SOFT files.
    """
    print("Using memory-efficient processing...")
    
    # First pass: identify matching samples
    matching_samples = set()
    sample_titles = {}
    
    with open(file_path, 'r', encoding='utf-8') as file:
        current_sample = None
        for line in file:
            line = line.strip()
            if line.startswith('^SAMPLE'):
                current_sample = line.split('=')[1].strip() if '=' in line else ''
            elif current_sample and line.startswith('!Sample_title'):
                title = line.split('=', 1)[1].strip() if '=' in line else ''
                sample_titles[current_sample] = title
                if title.startswith(sample_prefix):
                    matching_samples.add(current_sample)
    
    print(f"Found {len(matching_samples)} matching samples in first pass")
    
    if not matching_samples:
        return None
    
    # Second pass: extract expression data only for matching samples
    # Implementation would continue here for very large files...
    
    return extract_expression_by_prefix(file_path, sample_prefix)

# Example usage and testing
def main():
    # Configuration
    soft_file_path = "GSE45827_family.soft"  # Update this path
    sample_prefix = "Basal "  # Change this to your desired prefix
    
    print("Gene Expression Extractor for GEO SOFT Files")
    print("=" * 50)
    print(f"File: {soft_file_path}")
    print(f"Sample prefix: '{sample_prefix}'")
    print("=" * 50)
    
    # Extract expression data
    expression_matrix = extract_expression_by_prefix(soft_file_path, sample_prefix)
    
    if expression_matrix is not None:
        print("\n" + "="*50)
        print("EXTRACTION COMPLETED SUCCESSFULLY!")
        print("="*50)
        
        # Additional analysis
        print("\nSample columns:")
        for i, col in enumerate(expression_matrix.columns):
            print(f"{i+1:2d}. {col}")
        
        # Show some example genes
        print(f"\nFirst 10 genes/probes:")
        for i, gene in enumerate(expression_matrix.index[:10]):
            print(f"{i+1:2d}. {gene}")
    
    return expression_matrix

# Execute if run as script
if __name__ == "__main__":
    result = main()
    
    # Example: Extract different prefixes
    # Uncomment and modify as needed:
    # extract_expression_by_prefix("GSE45827_family.soft", "Luminal A", "luminal_a_expression")
    # extract_expression_by_prefix("GSE45827_family.soft", "Luminal B", "luminal_b_expression")

Gene Expression Extractor for GEO SOFT Files
File: GSE45827_family.soft
Sample prefix: 'Basal '
Parsing SOFT file: GSE45827_family.soft
Looking for samples with prefix: 'Basal '
------------------------------------------------------------
Found platform: GPL570
Processing line 10000...
Processing line 20000...
Processing line 30000...
Processing line 40000...
Processing line 50000...
Found matching sample: GSM1116084 - Basal Sample1 rep1
Processing line 60000...
Processing line 70000...
Processing line 80000...
Found matching sample: GSM1116085 - Basal Sample2 rep1-2
Processing line 90000...
Processing line 100000...
Processing line 110000...
Processing line 120000...
Processing line 130000...
Processing line 140000...
Found matching sample: GSM1116087 - Basal Sample4 rep1-2
Processing line 150000...
Processing line 160000...
Processing line 170000...
Processing line 180000...
Processing line 190000...
Processing line 200000...
Processing line 210000...
Processing line 220000...
Proces

/tmp/ipykernel_33472/940659731.py:223: RuntimeWarning: invalid value encountered in scalar divide
  print(f"\nMissing values: {missing_count}/{total_values} ({100*missing_count/total_values:.2f}%)")


In [9]:
# Extract Sample Titles from GEO SOFT File using BioPython
# Jupyter Notebook Code

# Import required libraries
from Bio import Geo
import pandas as pd

# Read the SOFT file
# Replace 'GSE45827_family.soft' with your actual file path
soft_file = "GSE45827_family.soft"

try:
    # Parse the SOFT file
    records = Geo.parse(open(soft_file))
    
    # Extract sample titles
    sample_titles = []
    
    for record in records:
        if record.entity_type == 'SAMPLE':
            sample_id = record.entity_id
            
            # Try different possible keys for sample names/titles
            title = None
            possible_keys = ['title', 'Sample_title', 'sample_title', 'description', 
                           'Sample_description', 'source_name_ch1', 'characteristics_ch1']
            
            for key in possible_keys:
                if key in record.entity_attributes and record.entity_attributes[key]:
                    # Get the full value, not just the first character
                    attr_value = record.entity_attributes[key]
                    if isinstance(attr_value, list) and attr_value:
                        title = attr_value[0]
                    elif isinstance(attr_value, str):
                        title = attr_value
                    break
            
            # If no title found, use the sample ID
            if not title:
                title = sample_id
            
            sample_titles.append({
                'Sample_ID': sample_id,
                'Title': title
            })
    
    # Display results
    print(f"Found {len(sample_titles)} samples:\n")
    
    # Create a DataFrame for better display
    df = pd.DataFrame(sample_titles)
    print(df.to_string(index=False))
    
    # Optional: Save to CSV
    # df.to_csv('sample_titles.csv', index=False)
    
except FileNotFoundError:
    print(f"File '{soft_file}' not found. Please check the file path.")
except Exception as e:
    print(f"Error parsing SOFT file: {e}")

# Alternative approach: Print all available attributes for debugging
print("\n" + "="*50)
print("Debug: Available attributes for first sample:")
print("="*50)

# Re-parse to show available attributes
records = Geo.parse(open(soft_file))
for record in records:
    if record.entity_type == 'SAMPLE':
        print(f"Sample ID: {record.entity_id}")
        print("Available attributes:")
        for key, value in record.entity_attributes.items():
            print(f"  {key}: {value}")
        break  # Just show first sample for debugging

Found 155 samples:

 Sample_ID                    Title
GSM1116084       Basal Sample1 rep1
GSM1116085     Basal Sample2 rep1-2
GSM1116086        Her2 Sample3 rep1
GSM1116087     Basal Sample4 rep1-2
GSM1116088      Her2 Sample5 rep1-2
GSM1116089      Her2 Sample6 rep1-2
GSM1116090       Basal Sample7 rep1
GSM1116091     Basal Sample8 rep1-2
GSM1116092       Basal Sample9 rep1
GSM1116093    Basal Sample10 rep1-2
GSM1116094      Basal Sample11 rep1
GSM1116095     Her2 Sample12 rep1-2
GSM1116096       Her2 Sample13 rep1
GSM1116097       Her2 Sample14 rep1
GSM1116098       Her2 Sample15 rep1
GSM1116099    Basal Sample16 rep1-2
GSM1116100       Her2 Sample17 rep1
GSM1116101      Basal Sample18 rep1
GSM1116102    Basal Sample19 rep1-2
GSM1116103       Her2 Sample20 rep1
GSM1116104     Her2 Sample21 rep1-2
GSM1116105       Her2 Sample22 rep1
GSM1116106    Basal Sample23 rep1-2
GSM1116107     Her2 Sample24 rep1-2
GSM1116108      Basal Sample25 rep1
GSM1116109      Basal Sample26 rep1
GSM11161

In [15]:
# Direct SOFT File Parser for Expression Data
# Alternative approach when BioPython table parsing doesn't work

import pandas as pd
import re
from collections import defaultdict

# Configuration
soft_file = "GSE45827_family.soft"
sample_substring = "Basal"  # Change this to your desired substring
output_csv = f"{sample_substring}_gene_expression.csv"

def parse_soft_file_directly(filename, substring):
    """
    Parse SOFT file directly to extract sample info and expression data
    """
    matching_samples = {}
    current_sample = None
    current_sample_title = None
    in_sample_table = False
    sample_data = defaultdict(dict)
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            
            # Check for sample start
            if line.startswith('^SAMPLE'):
                current_sample = line.split('=')[1].strip() if '=' in line else None
                current_sample_title = None
                in_sample_table = False
                
            # Get sample title/description
            elif current_sample and line.startswith('!Sample_title'):
                title = line.split('=')[1].strip() if '=' in line else ""
                if substring.lower() in title.lower():
                    current_sample_title = title
                    print(f"Found matching sample: {current_sample} - {title}")
                    
            # Check for table start
            elif current_sample and current_sample_title and line.startswith('!sample_table_begin'):
                in_sample_table = True
                continue
                
            # Check for table end
            elif line.startswith('!sample_table_end'):
                in_sample_table = False
                if current_sample and current_sample_title:
                    # Store the sample data
                    matching_samples[f"{current_sample}_{current_sample_title}"] = dict(sample_data[current_sample])
                
            # Parse table data
            elif in_sample_table and current_sample and current_sample_title:
                if line and not line.startswith('!') and not line.startswith('#'):
                    parts = line.split('\t')
                    if len(parts) >= 2:
                        gene_id = parts[0]
                        try:
                            expression_value = float(parts[1])
                            sample_data[current_sample][gene_id] = expression_value
                        except (ValueError, IndexError):
                            continue
    
    return matching_samples

# Parse the file
try:
    print(f"Parsing SOFT file for samples containing '{sample_substring}'...")
    expression_data = parse_soft_file_directly(soft_file, sample_substring)
    
    if not expression_data:
        print(f"No samples found containing substring '{sample_substring}'")
    else:
        print(f"\nFound {len(expression_data)} matching samples")
        
        # Get all unique gene IDs
        all_genes = set()
        for sample_data in expression_data.values():
            all_genes.update(sample_data.keys())
        all_genes = sorted(list(all_genes))
        
        print(f"Found {len(all_genes)} genes")
        
        # Create expression matrix
        expression_matrix = pd.DataFrame(index=all_genes, columns=expression_data.keys())
        
        # Fill in expression values
        for sample_name, sample_data in expression_data.items():
            for gene_id in all_genes:
                expression_matrix.loc[gene_id, sample_name] = sample_data.get(gene_id, None)
        
        # Save to CSV
        expression_matrix.to_csv(output_csv)
        print(f"\nExpression matrix saved to: {output_csv}")
        print(f"Matrix dimensions: {len(all_genes)} genes x {len(expression_data)} samples")
        
        # Display first few rows
        print("\nFirst 5 genes and first 3 samples:")
        print(expression_matrix.iloc[:5, :min(3, expression_matrix.shape[1])])
        
        # Show some statistics
        print(f"\nExpression value statistics:")
        print(f"Total values: {expression_matrix.count().sum()}")
        print(f"Missing values: {expression_matrix.isnull().sum().sum()}")

except FileNotFoundError:
    print(f"File '{soft_file}' not found. Please check the file path.")
except Exception as e:
    print(f"Error processing SOFT file: {e}")
    import traceback
    traceback.print_exc()

Parsing SOFT file for samples containing 'Basal'...
Found matching sample: GSM1116084 - Basal Sample1 rep1
Found matching sample: GSM1116085 - Basal Sample2 rep1-2
Found matching sample: GSM1116087 - Basal Sample4 rep1-2
Found matching sample: GSM1116090 - Basal Sample7 rep1
Found matching sample: GSM1116091 - Basal Sample8 rep1-2
Found matching sample: GSM1116092 - Basal Sample9 rep1
Found matching sample: GSM1116093 - Basal Sample10 rep1-2
Found matching sample: GSM1116094 - Basal Sample11 rep1
Found matching sample: GSM1116099 - Basal Sample16 rep1-2
Found matching sample: GSM1116101 - Basal Sample18 rep1
Found matching sample: GSM1116102 - Basal Sample19 rep1-2
Found matching sample: GSM1116106 - Basal Sample23 rep1-2
Found matching sample: GSM1116108 - Basal Sample25 rep1
Found matching sample: GSM1116109 - Basal Sample26 rep1
Found matching sample: GSM1116110 - Basal Sample27 rep1
Found matching sample: GSM1116111 - Basal Sample28 rep1
Found matching sample: GSM1116112 - Basal Sa

In [2]:
# Import necessary libraries
import pandas as pd
import re

def parse_soft_for_gene_ids(soft_file_path):
    """
    Parses a GEO SOFT file and extracts platform IDs with their corresponding Entrez Gene IDs.
    
    Args:
        soft_file_path (str): Path to the SOFT file
        
    Returns:
        pandas.DataFrame: A dataframe with columns 'ID' and 'ENTREZ_GENE_ID'
    """
    
    # Lists to store our results
    platform_ids = []
    entrez_gene_ids = []
    
    # Read the file
    with open(soft_file_path, 'r') as f:
        lines = f.readlines()
    
    # Variables to track when we're in the platform table section
    in_platform_table = False
    header = []
    
    for line in lines:
        line = line.strip()
        
        # Check if we're entering the platform table section
        if line.startswith('!platform_table_begin'):
            in_platform_table = True
            continue
        
        # Check if we're exiting the platform table section
        if line.startswith('!platform_table_end'):
            in_platform_table = False
            break
        
        # If we're in the platform table section
        if in_platform_table:
            # Check if this is the header line (starts with tab-separated field names)
            if line.startswith('ID\t') or line.startswith('ID '):
                header = line.split('\t')
                continue
            
            # Process data rows
            if header and line and not line.startswith('!'):
                fields = line.split('\t')
                if len(fields) >= len(header):
                    # Create a dictionary for this row
                    row_dict = dict(zip(header, fields))
                    
                    # Get the platform ID
                    platform_id = row_dict.get('ID', 'N/A')
                    platform_ids.append(platform_id)
                    
                    # Strategy 1: Try the standard field first
                    entrez_id = row_dict.get('ENTREZ_GENE_ID', None)
                    
                    # Strategy 2: If not found, try the common alternative 'GENE'
                    if not entrez_id or entrez_id == '':
                        entrez_id = row_dict.get('GENE', None)
                    
                    # Strategy 3: If still not found, try 'GeneID'
                    if not entrez_id or entrez_id == '':
                        entrez_id = row_dict.get('GeneID', None)
                    
                    # Strategy 4: If all else fails, try to parse 'gene_assignment'
                    if not entrez_id or entrez_id == '':
                        gene_assign = row_dict.get('gene_assignment', None)
                        if gene_assign and gene_assign != '':
                            # Split by '///' first to handle multiple assignments
                            assignments = gene_assign.split(" /// ")
                            for assignment in assignments:
                                # Split each assignment by ' // '
                                parts = assignment.split(" // ")
                                if len(parts) >= 5:  # Ensure there are enough parts
                                    potential_id = parts[4].strip()  # 5th element
                                    # Check if it looks like an Entrez ID (numeric and not empty)
                                    if potential_id and potential_id != '---' and potential_id.replace('-', '').isdigit():
                                        entrez_id = potential_id
                                        break  # Take the first valid one
                    
                    # Handle cases where no Entrez ID was found
                    if not entrez_id or entrez_id == '':
                        entrez_id = "N/A"
                    
                    entrez_gene_ids.append(entrez_id)
    
    # Create a DataFrame
    result_df = pd.DataFrame({
        'ID': platform_ids,
        'ENTREZ_GENE_ID': entrez_gene_ids
    })
    
    return result_df

# Alternative function if the file structure is different
def parse_soft_simple(soft_file_path):
    """
    Simpler parser that looks for ID and ENTREZ_GENE_ID patterns throughout the file.
    """
    platform_ids = []
    entrez_gene_ids = []
    current_id = None
    
    with open(soft_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            
            # Look for ID lines
            if line.startswith('ID'):
                parts = line.split('\t')
                if len(parts) > 1:
                    current_id = parts[1]
            
            # Look for Entrez Gene ID lines
            elif line.startswith('ENTREZ_GENE_ID') and current_id:
                parts = line.split('\t')
                if len(parts) > 1:
                    platform_ids.append(current_id)
                    entrez_gene_ids.append(parts[1])
                    current_id = None
            
            # Look for alternative field names
            elif (line.startswith('GENE\t') or line.startswith('GeneID\t')) and current_id:
                parts = line.split('\t')
                if len(parts) > 1 and parts[1].isdigit():
                    platform_ids.append(current_id)
                    entrez_gene_ids.append(parts[1])
                    current_id = None
    
    return pd.DataFrame({'ID': platform_ids, 'ENTREZ_GENE_ID': entrez_gene_ids})

# Usage example
if __name__ == "__main__":
    # Replace with your actual SOFT file path
    soft_file_path = "GSE45827_family.soft"  # or .txt
    
    try:
        print("Attempting to parse SOFT file...")
        
        # Try the main parser first
        result_dataframe = parse_soft_for_gene_ids(soft_file_path)
        
        if len(result_dataframe) == 0:
            print("Main parser found no data. Trying alternative parser...")
            result_dataframe = parse_soft_simple(soft_file_path)
        
        # Display the first few rows
        print("Extracted ID to Entrez Gene ID mapping:")
        print(result_dataframe.head(10))
        
        # Show some statistics
        print(f"\nTotal entries: {len(result_dataframe)}")
        valid_entries = len(result_dataframe[result_dataframe['ENTREZ_GENE_ID'] != 'N/A'])
        print(f"Entries with valid Entrez IDs: {valid_entries}")
        
        # Optional: Save to CSV
        # result_dataframe.to_csv('platform_to_entrez_mapping.csv', index=False)
        # print("Results saved to 'platform_to_entrez_mapping.csv'")
        
    except FileNotFoundError:
        print(f"Error: File not found at {soft_file_path}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

Attempting to parse SOFT file...
Extracted ID to Entrez Gene ID mapping:
          ID      ENTREZ_GENE_ID
0  1007_s_at   780 /// 100616237
1    1053_at                5982
2     117_at                3310
3     121_at                7849
4  1255_g_at                2978
5    1294_at  7318 /// 100847079
6    1316_at                7067
7    1320_at               11099
8  1405_i_at                6352
9    1431_at                1571

Total entries: 35177
Entries with valid Entrez IDs: 35002


In [5]:
# Import necessary libraries
import pandas as pd
import re

def parse_soft_for_gene_ids(soft_file_path):
    """
    Parses a GEO SOFT file and extracts platform IDs with their corresponding Entrez Gene IDs.
    For fields with multiple values (///), only the first value is taken.
    
    Args:
        soft_file_path (str): Path to the SOFT file
        
    Returns:
        pandas.DataFrame: A dataframe with columns 'ID' and 'ENTREZ_GENE_ID'
    """
    
    # Lists to store our results
    platform_ids = []
    entrez_gene_ids = []
    
    # Read the file
    with open(soft_file_path, 'r') as f:
        lines = f.readlines()
    
    # Variables to track when we're in the platform table section
    in_platform_table = False
    header = []
    
    for line in lines:
        line = line.strip()
        
        # Check if we're entering the platform table section
        if line.startswith('!platform_table_begin'):
            in_platform_table = True
            continue
        
        # Check if we're exiting the platform table section
        if line.startswith('!platform_table_end'):
            in_platform_table = False
            break
        
        # If we're in the platform table section
        if in_platform_table:
            # Check if this is the header line (starts with tab-separated field names)
            if line.startswith('ID\t') or line.startswith('ID '):
                header = line.split('\t')
                continue
            
            # Process data rows
            if header and line and not line.startswith('!'):
                fields = line.split('\t')
                if len(fields) >= len(header):
                    # Create a dictionary for this row
                    row_dict = dict(zip(header, fields))
                    
                    # Get the platform ID - take only the first value if multiple exist
                    platform_id = row_dict.get('ID', 'N/A')
                    if ' /// ' in platform_id:
                        platform_id = platform_id.split(' /// ')[0].strip()
                    platform_ids.append(platform_id)
                    
                    # Strategy 1: Try the standard field first - take only first value
                    entrez_id = row_dict.get('ENTREZ_GENE_ID', None)
                    if entrez_id and ' /// ' in entrez_id:
                        entrez_id = entrez_id.split(' /// ')[0].strip()
                    
                    # Strategy 2: If not found, try the common alternative 'GENE' - take only first value
                    if not entrez_id or entrez_id == '':
                        entrez_id = row_dict.get('GENE', None)
                        if entrez_id and ' /// ' in entrez_id:
                            entrez_id = entrez_id.split(' /// ')[0].strip()
                    
                    # Strategy 3: If still not found, try 'GeneID' - take only first value
                    if not entrez_id or entrez_id == '':
                        entrez_id = row_dict.get('GeneID', None)
                        if entrez_id and ' /// ' in entrez_id:
                            entrez_id = entrez_id.split(' /// ')[0].strip()
                    
                    # Strategy 4: If all else fails, try to parse 'gene_assignment' - take only first assignment
                    if not entrez_id or entrez_id == '':
                        gene_assign = row_dict.get('gene_assignment', None)
                        if gene_assign and gene_assign != '':
                            # Take only the FIRST assignment (before ///)
                            if ' /// ' in gene_assign:
                                gene_assign = gene_assign.split(' /// ')[0].strip()
                            
                            # Split the first assignment by ' // '
                            parts = gene_assign.split(" // ")
                            if len(parts) >= 5:  # Ensure there are enough parts
                                potential_id = parts[4].strip()  # 5th element
                                # Check if it looks like an Entrez ID (numeric and not empty)
                                if potential_id and potential_id != '---' and potential_id.replace('-', '').isdigit():
                                    entrez_id = potential_id
                    
                    # Handle cases where no Entrez ID was found
                    if not entrez_id or entrez_id == '':
                        entrez_id = "N/A"
                    
                    entrez_gene_ids.append(entrez_id)
    
    # Create a DataFrame
    result_df = pd.DataFrame({
        'ID': platform_ids,
        'ENTREZ_GENE_ID': entrez_gene_ids
    })
    
    return result_df

# Usage example
if __name__ == "__main__":
    # Replace with your actual SOFT file path
    # soft_file_path = "GSE45827_family.soft"
    soft_file_path = "GSE97562_family.soft"

    try:
        print("Attempting to parse SOFT file (taking first value from /// separators)...")
        
        result_dataframe = parse_soft_for_gene_ids(soft_file_path)
        
        # Display the first few rows
        print("Extracted ID to Entrez Gene ID mapping (first value only):")
        print(result_dataframe.head(15))
        
        # Show some statistics
        print(f"\nTotal entries: {len(result_dataframe)}")
        valid_entries = len(result_dataframe[result_dataframe['ENTREZ_GENE_ID'] != 'N/A'])
        print(f"Entries with valid Entrez IDs: {valid_entries}")
        
        # Show examples where multiple values were present
        multi_value_examples = result_dataframe[result_dataframe['ENTREZ_GENE_ID'].str.contains(' /// ', na=False)]
        if len(multi_value_examples) > 0:
            print(f"\nEntries that still contain multiple values (///): {len(multi_value_examples)}")
            print("Sample of multi-value entries:")
            print(multi_value_examples.head())
        
        # Optional: Save to CSV
        # result_dataframe.to_csv('platform_to_entrez_mapping_first_only.csv', index=False)
        # print("Results saved to 'platform_to_entrez_mapping_first_only.csv'")
        
    except FileNotFoundError:
        print(f"Error: File not found at {soft_file_path}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

Attempting to parse SOFT file (taking first value from /// separators)...
Extracted ID to Entrez Gene ID mapping (first value only):
         ID ENTREZ_GENE_ID
0   7896736            N/A
1   7896738            N/A
2   7896740          26682
3   7896742         728323
4   7896744         729759
5   7896746            N/A
6   7896748            N/A
7   7896750            N/A
8   7896752            N/A
9   7896754      100287497
10  7896756         400728
11  7896759         643837
12  7896761         148398
13  7896779         339451
14  7896798          84069

Total entries: 33297
Entries with valid Entrez IDs: 22037


In [11]:
import pickle
import pandas as pd
from collections import OrderedDict

def inspect_orange_metadata(metadata_path):
    """Inspect Orange's binary metadata file"""
    try:
        with open(metadata_path, 'rb') as f:
            metadata = pickle.load(f)
        
        print("=== ORANGE BINARY METADATA INSPECTION ===")
        print(f"Type of metadata object: {type(metadata)}")
        
        if isinstance(metadata, dict):
            print(f"\n--- METADATA KEYS ---")
            for key in metadata.keys():
                print(f"- {key} ({type(metadata[key])})")
                
            # Check for common Orange metadata structures
            if 'domain' in metadata:
                print(f"\n--- DOMAIN INFORMATION ---")
                domain = metadata['domain']
                if hasattr(domain, 'attributes'):
                    print(f"Attributes: {len(domain.attributes)}")
                    for i, attr in enumerate(domain.attributes[:3]):  # Show first 3
                        print(f"  {i}. {attr.name} ({type(attr).__name__})")
                        if hasattr(attr, 'attributes') and attr.attributes:
                            print(f"     Variable attributes: {list(attr.attributes.keys())}")
                
                if hasattr(domain, 'class_vars'):
                    print(f"Class variables: {len(domain.class_vars)}")
                
                if hasattr(domain, 'metas'):
                    print(f"Meta variables: {len(domain.metas)}")
                    for meta in domain.metas[:3]:
                        print(f"  - {meta.name} ({type(meta).__name__})")
            
            # Show other metadata
            print(f"\n--- ADDITIONAL METADATA ---")
            for key, value in metadata.items():
                if key != 'domain':
                    if hasattr(value, '__len__') and len(str(value)) > 100:
                        print(f"{key}: {type(value).__name__} with {len(value)} items")
                    else:
                        print(f"{key}: {value}")
                        
        elif isinstance(metadata, OrderedDict):
            print("Metadata is an OrderedDict")
            for key, value in metadata.items():
                print(f"{key}: {type(value)}")
                
    except Exception as e:
        print(f"Error reading metadata: {e}")
        import traceback
        traceback.print_exc()

# Usage
inspect_orange_metadata('GDS1326.csv.metadata')
print("\n")
inspect_orange_metadata('GDS2367.csv.metadata')
print("\n")
inspect_orange_metadata('GSE45827.csv.metadata')
print("\n")
inspect_orange_metadata('testando.csv.metadata')

=== ORANGE BINARY METADATA INSPECTION ===
Type of metadata object: <class 'dict'>

--- METADATA KEYS ---
- taxonomy_id (<class 'str'>)
- gene_as_attribute_name (<class 'bool'>)
- gene_id_column (<class 'str'>)

--- ADDITIONAL METADATA ---
taxonomy_id: 9606
gene_as_attribute_name: False
gene_id_column: Entrez ID


=== ORANGE BINARY METADATA INSPECTION ===
Type of metadata object: <class 'dict'>

--- METADATA KEYS ---
- taxonomy_id (<class 'str'>)
- gene_as_attribute_name (<class 'bool'>)
- gene_id_column (<class 'str'>)

--- ADDITIONAL METADATA ---
taxonomy_id: 9606
gene_as_attribute_name: False
gene_id_column: Entrez ID


=== ORANGE BINARY METADATA INSPECTION ===
Type of metadata object: <class 'dict'>

--- METADATA KEYS ---
- taxonomy_id (<class 'str'>)
- gene_as_attribute_name (<class 'bool'>)
- gene_id_column (<class 'str'>)

--- ADDITIONAL METADATA ---
taxonomy_id: 9606
gene_as_attribute_name: False
gene_id_column: Entrez ID


=== ORANGE BINARY METADATA INSPECTION ===
Type of metad